In [ ]:
# Install dependencies
!pip uninstall -y -q torch torchvision torchaudio cudf easyocr numba-cuda dask-cuda pylibraft-cu12 raft-dask-cu12 pylibcudf-cu12 rmm-cu12 quantecon cuvs-cu12 ucxx-cu12 cuml-cu12 cuda-python cuml libcuml libcuvs libraft numba ydata-profiling google-colab
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch>=2.1.0 transformers>=5.15.0 datasets>=2.14.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0 papermill>=2.7.0 testbook>=0.4.2
!pip install -q -U transformers


In [ ]:
import os
import sys
import time
import traceback
import torch
# Fix allocator fragmentation (exp16r OOM: reserved-but-unallocated
# grew to 1.1GB across alphas) — set BEFORE any CUDA use.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# KAGGLEHUB_CACHE: persist models across sessions
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/working/.model_cache"
os.makedirs("/kaggle/working/.model_cache", exist_ok=True)
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~/.kaggle_test")
sys.path.insert(0, WORKING)
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    device_L, device_S = "cuda:0", "cuda:1"
    dtype_L = dtype_S = torch.float32
elif num_gpus == 1:
    device_L = device_S = "cuda:0"
    dtype_L = torch.float16
    dtype_S = torch.float32
else:
    device_L = device_S = "cpu"
    dtype_L = dtype_S = torch.float32
if num_gpus > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | L={device_L}, S={device_S}")
RES = os.path.join(WORKING, "results")
os.makedirs(RES, exist_ok=True)
# Resolve models: kernel_sources (prev commit outputs) -> working -> HF
def resolve_model(hf_id):
    model_dir = "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b"
    # 1. Check each slug in /kaggle/input/ — listdir only (isdir lies on mounts)
    input_dir = "/kaggle/input"
    try:
        slugs = os.listdir(input_dir)
        print(f"resolve {hf_id}: slugs={slugs}")
        for slug in slugs:
            p_sub = os.path.join(input_dir, slug)
            try:
                contents = os.listdir(p_sub)
            except OSError:
                continue
            if model_dir in contents:
                p = os.path.join(p_sub, model_dir)
                print(f"Found {hf_id} at {p} (via {slug})")
                return p
    except Exception as e:
        print(f"DEBUG: /kaggle/input/ check failed: {e}")
    # 2. Check WORKING
    wf = os.path.join(WORKING, model_dir)
    if os.path.isdir(wf):
        print(f"Found {hf_id} at {wf}")
        return wf
    # 3. Return HF ID - will download + persist
    print(f"Need to download {hf_id} from HF")
    return hf_id
# Load + save for persistence (calls from_pretrained, then save_pretrained if from HF)
def load_model(cls, hf_id, **kw):
    path = resolve_model(hf_id)
    model = cls.from_pretrained(path, **kw)
    # Persist if loaded from HF
    if path == hf_id:
        dst = os.path.join(WORKING, "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b")
        if not os.path.isdir(dst):
            print(f"Saving {hf_id} to {dst}...")
            model.save_pretrained(dst)
            ensure_tokenizer(hf_id, dst)
            print(f"Saved")
    return model
# Ensure tokenizer files exist in a persisted model dir
def ensure_tokenizer(hf_id, dst):
    import transformers
    if not os.path.exists(os.path.join(dst, "tokenizer.json")):
        print(f"Saving tokenizer for {hf_id}...")
        transformers.AutoTokenizer.from_pretrained(hf_id).save_pretrained(dst)
SMALL = resolve_model("Qwen/Qwen3.5-0.8B")
LARGE = resolve_model("Qwen/Qwen3.5-4B")


In [ ]:
import torch
from transformers import DynamicCache
from transformers.cache_utils import (  # type: ignore[import-not-found]
    DynamicLayer,
    LinearAttentionLayer,
)
def compute_jacobian_for_tokens(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
) -> torch.Tensor:
    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.
    Args:
        model: Qwen3_5ForCausalLM instance.
        layer_idx: Layer index (0-based) to compute Jacobian at.
        input_ids: [1, S] token IDs.
        attention_mask: [1, S] attention mask.
        token_set: list of token IDs to compute gradients for.
    Returns:
        J: [|U|, D] Jacobian matrix (unnormalized).
    """
    model.eval()
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    # Match model dtype to avoid mismatch in decoder (fp16 models on GPU)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: the decoder forward must build a graph over the leaf
    # h AND the row loop below must run autograd.grad with grad mode ON,
    # even when the caller is inside torch.no_grad() (e.g. eval loops in
    # exp18/19). A no_grad caller otherwise gets "element 0 ... does not
    # have a grad_fn" — the rows loop is part of the same enable_grad block.
    with torch.enable_grad():
        logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
        J = _compute_jacobian_rows(logits, h, token_set)
    return J
def _run_prefix_pass(
    model, input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> DynamicCache:
    """Prefix pass: all tokens except last -> cache."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids[:, :-1],
            attention_mask=attention_mask[:, :-1],
            use_cache=True,
            output_hidden_states=False,
        )
    return outputs.past_key_values
def _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:
    """Full pass: extract hidden state at layer_idx for last token."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] = embeddings, [l+1] = output of layer l
    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]
    return h  # [1, 1, D]
def _deep_clone_cache(cache):
    """Deep clone cache, ensuring all tensors are independent copies.
    ponytail: copy.deepcopy works but is slow. Manual clone for speed + safety.
    """
    new_cache = DynamicCache()
    for idx in range(len(cache.layers)):
        layer = cache.layers[idx]
        if isinstance(layer, DynamicLayer):
            new_k = [
                k.clone() if k is not None else None
                for k in layer.keys  # type: ignore[union-attr]  # stub: list|None
            ]
            new_v = [
                v.clone() if v is not None else None
                for v in layer.values  # type: ignore[union-attr]
            ]
            new_cache.layers.append(DynamicLayer(keys=new_k, values=new_v))
        else:
            new_conv = {k: v.clone() for k, v in layer.conv_states.items()}
            new_recur = {k: v.clone() for k, v in layer.recurrent_states.items()}
            new_cache.layers.append(
                LinearAttentionLayer(conv_states=new_conv, recurrent_states=new_recur)
            )
    return new_cache
def _run_decoder_pass(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers using prefix cache."""
    # Clone cache tensors independently to avoid in-place mutation
    cache_clone = _deep_clone_cache(prefix_cache)
    text_model = model.model  # Qwen3_5TextModel
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    for i in range(start, n_layers):
        layer_type = model.config.layer_types[i]
        attn_mask = causal_mask_mapping[layer_type]
        x = text_model.layers[i](
            x,
            position_embeddings=position_embeddings,
            attention_mask=attn_mask,
            position_ids=text_position_ids,
            past_key_values=cache_clone,
        )
    x = text_model.norm(x)
    logits = model.lm_head(x[:, -1, :])
    return logits
def _build_causal_masks(text_model, text_pos, cache, x):
    """Build causal mask mapping for a single token with cache."""
    from transformers.masking_utils import (
        create_causal_mask,
        create_recurrent_attention_mask,
    )
    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)
    mask_kwargs = {
        "config": text_model.config,
        "inputs_embeds": dummy_embeds,
        "attention_mask": None,
        "past_key_values": cache,
        "position_ids": text_pos,
    }
    return {
        "full_attention": create_causal_mask(**mask_kwargs),
        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),
    }
def _compute_jacobian_rows(
    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]
) -> torch.Tensor:
    """Compute Jacobian rows via loop over token_set."""
    logits_subset = logits[0, token_set]  # [|U|]
    D = h.shape[-1]
    J = torch.zeros(len(token_set), D, device=h.device, dtype=h.dtype)
    for i in range(len(token_set)):
        grad = torch.autograd.grad(
            outputs=logits_subset[i],
            inputs=h,
            retain_graph=(i < len(token_set) - 1),
        )[0]
        J[i] = grad[0, 0, :]
    return J
def softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Gauge-invariant Jacobian: J^(p) = diag(p) - pp^T @ J.
    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k
    Removes the common-mode shift (gauge) from raw Jacobian.
    """
    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]
    return p[:, None] * (J - J_bar)  # [K, D]
def uniform_centered(J: torch.Tensor) -> torch.Tensor:
    """Uniform centering: J - mean(J). Removes average shift."""
    return J - J.mean(dim=0, keepdim=True)
def _last_layer_jacobian(h: torch.Tensor, norm: torch.nn.Module) -> torch.Tensor:
    """Analytical J = d h_norm / d h at the final norm (no backward passes).
    Both Qwen3_5RMSNorm and nn.LayerNorm are handled. J is evaluated at the
    norm INPUT (h), never at h_norm — evaluating the formula at the norm
    output gives a matrix with ~0.87 relative error (no longer LayerNorm-like
    statistics). Qwen3.5's final norm is Qwen3_5RMSNorm:
      h_norm = (h * rsqrt(mean(h^2) + eps) * (1 + w))
      J = r * (diag(1+w) - xhat x^T / D),  r = rsqrt(mean(h^2)+eps), xhat = r*h
    LayerNorm: J = (I - 1/D - xhat xhat^T / D) / std, rows scaled by w.
    """
    x = h.reshape(1, -1).float()  # [1, D], the norm INPUT (row vector)
    D = x.shape[-1]
    eps = float(getattr(norm, "eps", None) or 1e-5)
    w = getattr(norm, "weight", None)
    dev = x.device
    if getattr(norm, "bias", None) is not None:  # LayerNorm family
        mu = x.mean(dim=-1, keepdim=True)
        std = (x.var(dim=-1, unbiased=False, keepdim=True) + eps).sqrt()
        xhat = (x - mu) / std  # [1, D]
        J = (
            torch.eye(D, device=dev)
            - torch.ones(D, D, device=dev) / D
            - xhat.T @ xhat / D
        ) / std.squeeze(0)
        if w is not None:
            J = J * w.float().squeeze(0).unsqueeze(1)  # J = diag(w) @ J
        return J
    # RMSNorm family (Qwen3_5RMSNorm, Llama-style).
    r = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    xhat = x * r  # [D]
    # y_j = (1+w_j) r x_j; J = r * diag(1+w) @ (I - xhat xhat^T / D)
    # (derived: J_ij = g_j (r d_ij - r^3 x_i x_j / D) = g_j r (d_ij - xhat_i xhat_j / D))
    g = 1.0 + w.float().squeeze(0) if w is not None else None
    M = torch.eye(D, device=dev) - xhat.T @ xhat / D
    if g is None:
        return r.squeeze(0) * M
    return r.squeeze(0) * torch.diag(g) @ M
def compute_hidden_jacobian(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
) -> torch.Tensor:
    """Compute Jacobian of normalized last hidden state w.r.t. hidden state at layer_idx.
    J_norm = d(h_norm) / d(h_{layer_idx}), shape [D, D].
    Independent of lm_head — computed once, reused across W updates.
    Then J_S = lm_head.weight[token_set] @ J_norm is differentiable through W.
    For last_layer: J_norm is LayerNorm Jacobian (analytical, no backward passes).
    Args:
        create_graph: if True, J_norm has grad_fn.
                      h is kept connected to model params (no detach). Full forward pass.
                      Costs ~D backward passes per sample.
        verbose: print progress during J_norm computation.
    """
    model.eval()
    n_layers = model.config.num_hidden_layers
    if create_graph:
        # Keep gradient graph alive: h is connected to model params (+ any hooks)
        S = input_ids.shape[1]
        # Build prefix cache FIRST — before any forward involving hooks.
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
        # Full forward to get h at layer_idx — connected to all model params + hooks
        with torch.enable_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
        h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D], connected!
        model_dtype = next(model.parameters()).dtype
        h = h.to(model_dtype)  # preserve grad graph
        if h.grad_fn is None:
            # Fully frozen graph: no trainable path reaches this layer
            # (e.g. linear_attention pair with no LoRA patch, or a
            # no_grad caller). Make h a leaf so the decoder pass below
            # still yields a differentiable h_norm; J_norm values are
            # identical to the connected case.
            h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            with torch.enable_grad():
                h_norm = model.model.norm(h[:, -1, :])  # [1, D]
        else:
            with torch.enable_grad():
                h_norm = _run_decoder_to_norm(
                    model.model, prefix_cache, h, S, layer_idx
                )
        # [1, 1, D] — _compute_jacobian_rows_full indexes grad[0, 0, :].
        # enable_grad: autograd.grad needs grad mode ON even when the
        # caller is inside torch.no_grad().
        with torch.enable_grad():
            J_norm = _compute_jacobian_rows_full(
                h_norm, h, create_graph=True, verbose=verbose
            )
        return J_norm
    # Last layer: analytical norm Jacobian, no backward passes.
    if layer_idx == n_layers - 1:
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
        return _last_layer_jacobian(h, model.model.norm)
    # General case: decode through remaining layers + norm
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: h_norm must carry a grad_fn over h even if the caller
    # is inside torch.no_grad() (e.g. eval loops in exp16r/exp17). The row
    # loop below (autograd.grad) ALSO needs grad mode ON — a no_grad caller
    # otherwise gets "element 0 ... does not have a grad_fn".
    with torch.enable_grad():
        h_norm = _run_decoder_to_norm(model.model, prefix_cache, h, S, layer_idx)
        J_norm = _compute_jacobian_rows_full(
            h_norm, h, create_graph=create_graph, verbose=verbose
        )
    return J_norm
def _run_decoder_to_norm(
    text_model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers, return norm(x) without lm_head."""
    cache_clone = _deep_clone_cache(prefix_cache)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = text_model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    for i in range(start, n_layers):
        layer_type = text_model.config.layer_types[i]
        attn_mask = causal_mask_mapping[layer_type]
        x = text_model.layers[i](
            x,
            position_embeddings=position_embeddings,
            attention_mask=attn_mask,
            position_ids=text_position_ids,
            past_key_values=cache_clone,
        )
    return text_model.norm(x[:, -1, :])
def _compute_jacobian_rows_full(
    output: torch.Tensor,
    h: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
) -> torch.Tensor:
    """Full [D, D] Jacobian via autograd.grad."""
    import time as _time
    D = h.shape[-1]
    out_flat = output.flatten()
    grads = []
    # create_graph=True: every row must retain the graph — the caller
    # backpropagates a final loss through J_norm afterwards (exp17 bridge
    # loss -> o_proj HVP). retain=(i<D-1) would free the (second-order)
    # graph after the last row, so the caller's first backward would fail
    # with "backward through the graph a second time". Memory is bounded
    # by the refresh schedule (one graph per si, freed at the next refresh
    # / pair end).
    start = _time.time()
    for i in range(D):
        grad = torch.autograd.grad(
            outputs=out_flat[i],
            inputs=h,
            retain_graph=(True if create_graph else i < D - 1),
            create_graph=create_graph,
        )[0]
        grads.append(grad[0, 0, :])
        if verbose and i % 128 == 0:
            elapsed = _time.time() - start
            rate = i / elapsed if elapsed > 0 else 999
            eta = elapsed * (D / i) - elapsed if i > 0 else 0.0
            print(f"        J_norm: {i}/{D} ({rate:.0f}/s, eta={eta:.0f}s)")
    J = torch.stack(grads)
    elapsed_total = _time.time() - start
    if verbose:
        print(f"        J_norm done: {D}/{D} in {elapsed_total:.1f}s")
    return J


In [ ]:
import hashlib
import os
import torch
_gramian_cache: dict[str, torch.Tensor] = {}
_GRAMIAN_DIR = os.path.join("results", ".gramians")
def jacobian_bridge(
    J_L: torch.Tensor,
    J_S: torch.Tensor,
    E_L: torch.Tensor,
    E_S: torch.Tensor,
    probs_L: torch.Tensor,
    threshold: float = 1e-6,
) -> float:
    """Compute Jacobian bridge loss between large and small models.
    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.
    The inverse Gramian (E_S^T @ E_S)^{-1} corrects for non-orthogonal
    embeddings, so identity (J_L == J_S, E_L == E_S) yields loss ~0.
    """
    G_inv = _inverse_gramian(E_S)
    J_L_proj = _project_to_vocab(J_L, E_L)
    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)
    mask = _valid_mask(J_S, J_S_target, threshold)
    if mask.sum() == 0:
        return 1.0
    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)
def _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:
    """Project J_L [|U|, D_L] through E_L^T -> [|U|, V]."""
    return J_L.float() @ E_L.float().T
def _project_to_small(
    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor
) -> torch.Tensor:
    """Project [|U|, V] through E_S @ G_inv -> [|U|, D_S].
    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.
    """
    return J_L_proj @ E_S.float() @ G_inv
def _inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""
    key = _gramian_key(E)
    if key not in _gramian_cache:
        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")
        if os.path.exists(path):
            _gramian_cache[key] = torch.load(path, weights_only=True)
        else:
            _gramian_cache[key] = _compute_inverse_gramian(E)
            os.makedirs(_GRAMIAN_DIR, exist_ok=True)
            torch.save(_gramian_cache[key], path)
    return _gramian_cache[key].to(E.device)
def _gramian_key(E: torch.Tensor) -> str:
    """Stable hash of an embedding matrix for disk caching.
    Uses first/last rows as representative content — hashing the full
    248k×1024 matrix would be prohibitively expensive.
    """
    header = f"{tuple(E.shape)}|{str(E.device)}".encode()
    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()
    h = hashlib.sha256(data).hexdigest()[:16]
    return f"g_{h}"
def _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""
    G = E.float().T @ E.float()
    return torch.linalg.pinv(G)
def _valid_mask(
    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float
) -> torch.Tensor:
    """Mask rows where either norm < threshold."""
    n1 = J_S.float().norm(dim=1)
    n2 = J_target.float().norm(dim=1)
    return (n1 > threshold) & (n2 > threshold)
def _weighted_cosine_loss(
    J_S: torch.Tensor,
    J_target: torch.Tensor,
    probs: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    """Weighted cosine dissimilarity loss."""
    cos_sim = torch.nn.functional.cosine_similarity(
        J_S.float()[mask], J_target[mask], dim=1
    )
    w = probs[mask]
    w = w / (w.sum() + 1e-8)
    return (w * (1.0 - cos_sim)).sum().item()


In [ ]:
import torch
def compute_esg(
    jacobians: list[torch.Tensor],
    n_components: int = 32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute ESG from a list of Jacobian matrices.
    Args:
        jacobians: list of N Jacobian matrices, each [|U|, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: [n_components] sorted descending.
        eigenvectors: [n_components, |U|] corresponding eigenvectors.
    """
    G = _build_gramian(jacobians)
    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)
    return eigenvalues, eigenvectors
def _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:
    """G = (1/N) * sum_i (J_i @ J_i^T)."""
    U = jacobians[0].shape[0]
    device = jacobians[0].device
    G = torch.zeros(U, U, device=device, dtype=torch.float)
    for J in jacobians:
        G += J.float() @ J.float().T
    G /= len(jacobians)
    return G
def _spectral_decomposition(
    G: torch.Tensor, n_components: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigen decomposition, sorted descending."""
    eigenvalues, eigenvectors = torch.linalg.eigh(G)
    # Sort descending
    idx = eigenvalues.argsort(descending=True)
    eigenvalues = eigenvalues[idx][:n_components]
    eigenvectors = eigenvectors[:, idx][:, :n_components].T
    return eigenvalues, eigenvectors


In [ ]:
from collections import Counter
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
def uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:
    """Uniform layer mapping from large to small model layers."""
    mapping = {}
    for i in range(n_layers_large):
        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))
        mapping[i] = j
    return mapping
def get_layer_types(model: AutoModelForCausalLM) -> list[str]:
    """Get layer types from model config (full_attention / linear_attention)."""
    return list(model.config.layer_types)  # type: ignore[attr-defined]
_DOMAIN_DATASETS = {
    "wiki": ("allenai/c4", "en", "train", "text"),
    "math": ("openai/gsm8k", "main", "train", "question"),
    "code": ("neyralabs/apex-agents", None, "train", "prompt"),
    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),
}
def _load_dataset_for_domain(
    domain: str,
) -> tuple[str, str | None, str, str]:
    """Return (dataset_id, config, split, text_field) for a domain.
    Falls back to wiki (C4) if the target dataset is unavailable.
    """
    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]
    try:
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        load_dataset(ds_id, **kw)  # type: ignore[call-overload]
        return ds_id, cfg, split, field
    except Exception:
        print(f"  Warning: domain {domain} ({ds_id}) unavailable, falling back to C4")
        return _DOMAIN_DATASETS["wiki"]
def _extract_text_from_sample(
    sample: dict,
    field: str,
) -> str:
    """Extract text from a dataset sample, handling different formats."""
    if field == "messages":
        # Chat format: list of {role, content}
        texts = [m.get("content", "") for m in sample.get("messages", [])]
        return " ".join(texts)
    val = sample.get(field, "")
    if isinstance(val, list):
        # e.g. code_contests solutions: list of strings
        return "\n".join(str(v) for v in val)
    if isinstance(val, dict):
        return str(val)
    return val
def load_calibration_data(
    model_name: str,
    max_length: int = 512,
    min_length: int = 256,
    n_samples: int = 4,
    domain: str = "wiki",
) -> list[dict[str, torch.Tensor]]:
    """Load and tokenize calibration data.
    domain: 'wiki' (C4), 'math' (GSM8K), 'code' (apex-agents), 'chat' (ultrachat).
    Falls back to wiki if the target dataset is unavailable.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds_id, cfg, split, field = _load_dataset_for_domain(domain)
    kw: dict[str, object] = {"split": split, "streaming": True}
    if cfg is not None:
        kw["name"] = cfg
    dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
    samples = []
    for raw in dataset:
        text = _extract_text_from_sample(raw, field)
        if not text:
            continue
        tokens = tokenizer(
            text, max_length=max_length, truncation=True, return_tensors="pt"
        )
        if tokens["input_ids"].shape[1] >= min_length:
            samples.append(tokens)
        if len(samples) >= n_samples:
            break
    return samples
def build_global_token_set(
    model: AutoModelForCausalLM,
    calibration_samples: list[dict[str, torch.Tensor]],
    K: int = 64,
) -> list[int]:
    """Build global token set from top-K logits across all samples."""
    token_counter: Counter[int] = Counter()
    model.eval()  # type: ignore[attr-defined]
    device = next(model.parameters()).device  # type: ignore[attr-defined]
    with torch.no_grad():
        for sample in calibration_samples:
            inputs = {
                "input_ids": sample["input_ids"].to(device),
                "attention_mask": sample["attention_mask"].to(device),
            }
            outputs = model(**inputs)  # type: ignore[operator]
            logits = outputs.logits[:, -1, :]
            top_idx = logits.topk(K, dim=-1).indices[0]
            token_counter.update(top_idx.tolist())
    U = [tid for tid, _ in token_counter.most_common(K)]
    return sorted(U)
def load_models(
    large_name: str, small_name: str, dtype: torch.dtype = torch.float16
) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Load large and small models with eager attention."""
    kwargs = {
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)
    large.eval()  # type: ignore[attr-defined]
    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)
    small.eval()  # type: ignore[attr-defined]
    return large, small
def extract_embeddings(
    model: AutoModelForCausalLM,
) -> torch.Tensor:
    """Extract embedding matrix from model."""
    return model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]


In [ ]:
import os
# Max depth under input_root to search (producer puts cache at depth 2:
# results/targets; 4 covers any reasonable layout).
_MAX_DEPTH = 4
def _has_pt(d: str) -> bool:
    try:
        return any(f.endswith(".pt") for f in os.listdir(d))
    except OSError:
        return False
def _walk(root: str, depth: int):
    """Yield dirs under root (listdir-based), depth-limited."""
    try:
        entries = sorted(os.listdir(root))
    except OSError:
        return
    for e in entries:
        p = os.path.join(root, e)
        if _has_pt(p):
            yield p
        if depth > 1:
            yield from _walk(p, depth - 1)
def find_targets_dir(preferred: str, input_root: str = "/kaggle/input") -> str:
    """Return the dir holding the target-cache *.pt files.
    Order: `preferred` (local path / runner-injected), then a listdir walk
    of every mount under `input_root`. Raises FileNotFoundError with the
    mount inventory when nothing is found (no silent fallback, no listdir crash).
    """
    if _has_pt(preferred):
        return preferred
    for hit in _walk(input_root, _MAX_DEPTH):
        return hit
    mounts = []
    try:
        for d in sorted(os.listdir(input_root)):
            try:
                mounts.append(f"{d}/:{os.listdir(os.path.join(input_root, d))[:4]}")
            except OSError:
                mounts.append(f"{d}/:<unreadable>")
    except OSError:
        mounts.append("<no input mount>")
    raise FileNotFoundError(
        f"No target cache .pt in {preferred} nor under {input_root}/ "
        f"(mounts: {mounts}). Run Stage 0 first — it must publish results/targets/*.pt."
    )


In [ ]:
from __future__ import annotations
def _uniform_layer_mapping(n_L: int, n_S: int) -> dict[int, int]:
    """Uniform large->small layer mapping: l_L -> round(l_L*(n_S-1)/(n_L-1)).
    Inlined copy of prism.calibration.uniform_layer_mapping (identical formula)
    so this module stays standalone for kernel inlining.
    """
    return {
        i: round(i * (n_S - 1) / (n_L - 1)) for i in range(n_L)
    }
def get_layer_types(model) -> list[str]:
    """Per-layer attention type from model.config.layer_types.
    Qwen3.5 style hybrid: entries 'full_attention' or 'linear_attention'.
    `model.config` (post-AutoModel load) exposes `layer_types` at top level
    (Qwen3_5TextConfig). Returns lowercase list.
    """
    types = getattr(model.config, "layer_types", None)
    if types is None:
        # Fallback: nested text_config (defensive; Qwen3_5 exposes top-level).
        tc = getattr(model.config, "text_config", None)
        types = getattr(tc, "layer_types", None)
    if types is None:
        raise RuntimeError(
            "model.config has no layer_types (and no text_config.layer_types)"
        )
    return [str(t) for t in types]
def select_pairs(n_L: int, n_S: int, layer_types: list[str]) -> list[tuple[int, int]]:
    """Return (l_L, l_S) pairs: full-attn l_S + 0_0 + last, sorted by l_S."""
    if n_L < 1 or n_S < 1 or len(layer_types) != n_S:
        raise ValueError(f"invalid: n_L={n_L}, n_S={n_S}, layer_types={len(layer_types)}")
    mapping = _uniform_layer_mapping(n_L, n_S)
    full_S = {i for i, t in enumerate(layer_types) if t == "full_attention"}
    selected = {0, n_S - 1} | full_S
    pairs = [_nearest_preimage(mapping, j) for j in sorted(selected)]
    if (n_L - 1, n_S - 1) not in pairs:
        pairs.append((n_L - 1, n_S - 1))
    return pairs
def _nearest_preimage(mapping: dict[int, int], j: int) -> tuple[int, int]:
    """l_L = argmin |mapping[l] - j|, tie -> smaller l (the selected l_S is j)."""
    best_l, best_d = 0, float("inf")
    for li in range(len(mapping)):
        d = abs(mapping[li] - j)
        if d < best_d:
            best_l, best_d = li, d
    return (best_l, j)
def classify_pair(l_S: int, layer_types: list[str]) -> str:
    """'bridge' if small-model layer l_S has o_proj (full_attention), else 'kl_only'."""
    return (
        "bridge"
        if l_S < len(layer_types) and layer_types[l_S] == "full_attention"
        else "kl_only"
    )


In [ ]:
import json
import os
import time
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM  # type: ignore[import-not-found]
def _resolve_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"
def _gpu_config(device: str) -> dict:
    """Return parameter config adapted to device."""
    if device.startswith("cuda"):
        return {
            "n_samples": 4,
            "K": 256,
            "dtype_L": torch.bfloat16,
            "dtype_S": torch.float32,
        }
    return {
        "n_samples": 2,
        "K": 128,
        "dtype_L": torch.float32,
        "dtype_S": torch.float32,
    }
def _save_checkpoint(targets_dir: str, pairs: list[tuple], done_pairs: list[tuple]):
    """Save progress checkpoint."""
    ckpt = {
        "pairs": [[a, b] for a, b in pairs],
        "done": [[a, b] for a, b in done_pairs],
    }
    with open(os.path.join(targets_dir, ".checkpoint.json"), "w") as f:
        json.dump(ckpt, f)
def main_16stage0targets(
    large_model: str = "/data/Qwen/Qwen3.5-4B",
    small_model: str = "Qwen/Qwen3.5-0.8B",
    device_L: str = "auto",
    device_S: str = "auto",
    max_length: int = 64,
    min_length: int = 32,
    targets_dir: str = "results/targets",
    results_dir: str = "results",
):
    torch.manual_seed(42)
    start = time.time()
    if device_L == "auto":
        device_L = "cuda" if torch.cuda.is_available() else "cpu"
    if device_S == "auto":
        device_S = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = _gpu_config(device_L)  # use device_L for large model config
    if device_L == "cpu" and device_S == "cpu":
        torch.set_num_threads(4)
    print(f"Device L: {device_L}, S: {device_S}")
    print(
        f"Config: n_samples={cfg['n_samples']}, K={cfg['K']}, "
        f"dtype_L={cfg['dtype_L']}, dtype_S={cfg['dtype_S']}"
    )
    os.makedirs(targets_dir, exist_ok=True)
    # -- Load models -------------------------------------------------------
    print(f"Loading large model: {large_model}")
    mL = AutoModelForCausalLM.from_pretrained(
        large_model,
        torch_dtype=cfg["dtype_L"],
        attn_implementation="eager",
        low_cpu_mem_usage=True,
    ).to(device_L)  # type: ignore[arg-type]
    mL.eval()
    print(f"Loading small model: {small_model}")
    mS = AutoModelForCausalLM.from_pretrained(
        small_model,
        torch_dtype=cfg["dtype_S"],
        attn_implementation="eager",
        low_cpu_mem_usage=True,
    ).to(device_S)  # type: ignore[arg-type]
    mS.eval()
    n_L = mL.config.num_hidden_layers
    n_S = mS.config.num_hidden_layers
    print(f"Large: {n_L} layers, {mL.config.hidden_size} hidden")
    print(f"Small: {n_S} layers, {mS.config.hidden_size} hidden")
    # -- Re-selection (contract 0.1/0.2): per l_S = full-attn + 0_0 + last --
    layer_types = get_layer_types(mS)  # type: ignore[arg-type]
    pairs = select_pairs(n_L, n_S, layer_types)
    print(f"layer_types: {layer_types}")
    print(f"Selected pairs ({len(pairs)}): {pairs}")
    # -- Calibration data --------------------------------------------------
    print(f"Loading {cfg['n_samples']} calibration samples...")
    samples = load_calibration_data(
        small_model,
        max_length=max_length,
        min_length=min_length,
        n_samples=cfg["n_samples"],
    )
    # -- Embeddings --------------------------------------------------------
    E_L = extract_embeddings(mL).to(device_L)  # [V, D_L]  # type: ignore[arg-type]
    E_S = extract_embeddings(mS).to(device_S)  # [V, D_S]  # type: ignore[arg-type]
    print(f"E_L: {E_L.shape}, E_S: {E_S.shape}")
    # -- G_S_inv = (E_S^T E_S)^{-1} float64 for invertibility ------------
    print("Computing G_S_inv = (E_S^T E_S)^{-1}...")
    G_S = E_S.float().T @ E_S.float()  # [D_S, D_S]
    G_S_inv = torch.linalg.pinv(G_S.double()).float()  # [D_S, D_S]
    print(f"G_S_inv shape: {G_S_inv.shape}")
    # -- Token set from first sample (large model top-K) ------------------
    ids0 = samples[0]["input_ids"].to(device_L)
    attn0 = samples[0]["attention_mask"].to(device_L)
    with torch.no_grad():
        lg0 = mL(input_ids=ids0, attention_mask=attn0).logits[:, -1, :]
    token_set = sorted(lg0.topk(cfg["K"], dim=-1).indices[0].cpu().tolist())
    print(f"Token set size: {len(token_set)}")
    # -- Check existing cache ----------------------------------------------
    # A pair is "done" only if every sample file exists AND has the v2
    # schema (p_L_full, input_ids). v1 files (pre-pairing era) are recomputed.
    done_pairs: list[tuple[int, int]] = []
    for pair in pairs:
        pair_key = f"{pair[0]}_{pair[1]}"
        ok = True
        for si in range(cfg["n_samples"]):
            path = os.path.join(targets_dir, f"{pair_key}_{si}.pt")
            if not os.path.exists(path):
                ok = False
                break
            try:
                d = torch.load(path, weights_only=True)
            except Exception:
                ok = False
                break
            if "p_L_full" not in d or "input_ids" not in d:
                ok = False
                break
        if ok:
            done_pairs.append(pair)
            print(f"  Pair {pair_key}: cached, skipping")
        else:
            print(f"  Pair {pair_key}: recompute (missing or pre-v2 cache)")
    # -- Check checkpoint --------------------------------------------------
    ckpt_path = os.path.join(targets_dir, ".checkpoint.json")
    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            ckpt = json.load(f)
        for d in ckpt.get("done", []):
            dp = (d[0], d[1])
            if dp not in done_pairs:
                done_pairs.append(dp)
    remaining = [p for p in pairs if p not in done_pairs]
    print(f"Remaining pairs: {len(remaining)}/{len(pairs)}")
    if not remaining:
        print("All pairs already cached. Stage 0 complete.")
        print(f"Total time: {time.time() - start:.0f}s")
        return
    # -- Runtime guard: measure first Jacobian time ------------------------
    _guard_checked = False
    _guard_k = cfg["K"]
    _guard_n = cfg["n_samples"]
    # -- Compute targets ---------------------------------------------------
    for l_L, l_S in remaining:
        pair_key = f"{l_L}_{l_S}"
        print(f"\n=== Pair {pair_key} ===")
        # -- Runtime guard: measure first Jacobian time --------------------
        # GPU: ~0.5s per token-row is normal (K=256 -> ~135s on T4); the
        # threshold only fires on CPU or a broken CUDA environment.
        _guard_thresh = 300 if device_L.startswith("cuda") else 60
        _probe_J = None
        if not _guard_checked:
            _guard_checked = True
            print(f"  Probing J_L (layer {l_L}) timing...")
            t0 = time.time()
            _probe_J = compute_jacobian_for_tokens(mL, l_L, ids0, attn0, token_set)
            t_jL = time.time() - t0
            if t_jL > _guard_thresh:
                _probe_J = None
                print(
                    f"  WARNING: Jacobian took {t_jL:.1f}s (>{_guard_thresh}s). "
                    f"Reducing K and n_samples."
                )
                _guard_k = max(32, cfg["K"] // 2)
                _guard_n = max(1, cfg["n_samples"] // 2)
                # Rebuild token set with smaller K
                token_set = sorted(lg0.topk(_guard_k, dim=-1).indices[0].cpu().tolist())
                print(f"  New K={_guard_k}, n_samples={_guard_n}")
            else:
                print(f"  Jacobian time: {t_jL:.1f}s — OK, reused for sample 1")
        for si, s in enumerate(samples[:_guard_n]):
            ids_s = s["input_ids"].to(device_S)
            attn_s = s["attention_mask"].to(device_S)
            ids_L_s = ids_s.to(device_L)
            attn_L_s = attn_s.to(device_L)
            # -- J_L + transport for THIS sample --------------------------
            if _probe_J is not None and si == 0:
                # Probe computed exactly this (same model, same sample 0).
                J_L_s = _probe_J
                _probe_J = None
            else:
                print(f"  Computing J_L (layer {l_L}, sample {si + 1})...")
                J_L_s = compute_jacobian_for_tokens(
                    mL, l_L, ids_L_s, attn_L_s, token_set
                )
            with torch.no_grad():
                logits_L_s = mL(input_ids=ids_L_s, attention_mask=attn_L_s).logits[
                    :, -1, :
                ]
                p_s = torch.softmax(logits_L_s[0, token_set], dim=-1).to(device_S)
            print(f"  Transporting (sample {si + 1})...")
            J_L_proj_s = (
                J_L_s.float().to(device_S) @ E_L.float().to(device_S).T
            )  # [|U|, V]
            J_tgt_s = J_L_proj_s @ E_S.float() @ G_S_inv  # [|U|, D_S]
            J_tgt_p_s = softmax_jacobian(J_tgt_s, p_s)  # [|U|, D_S]
            # -- J_S_p on this sample ------------------------------------
            print(f"  Computing J_S (layer {l_S}, sample {si + 1})...")
            J_S = compute_jacobian_for_tokens(mS, l_S, ids_s, attn_s, token_set)
            J_S_p = softmax_jacobian(J_S, p_s)
            # Cache
            cache_path = os.path.join(targets_dir, f"{pair_key}_{si}.pt")
            torch.save(
                {
                    "J_tgt_p": J_tgt_p_s.cpu(),
                    "J_S_p": J_S_p.cpu(),
                    "p": p_s.cpu(),
                    # Full-vocab p_L for THIS sample: exp18 needs per-sample
                    # KL(p_ed||p_L) without a layer-mismatched J target.
                    "p_L_full": F.softmax(logits_L_s[0], dim=-1).float().cpu(),
                    "pair": pair_key,
                    "l_S": l_S,
                    "l_L": l_L,
                    "token_set": token_set,
                    "K": len(token_set),
                    # Per-sample ids: lets consumers verify sample<->target
                    # pairing across kernel runs (no silent position pairing).
                    "input_ids": ids_s.cpu(),
                },
                cache_path,
            )
            print(f"  Saved: {cache_path}")
            del J_L_s, J_L_proj_s, J_tgt_s, J_tgt_p_s, J_S, J_S_p
        done_pairs.append((l_L, l_S))
        _save_checkpoint(targets_dir, pairs, done_pairs)
        print(f"  Pair {pair_key} done. Elapsed: {time.time() - start:.0f}s")
    # -- Summary -----------------------------------------------------------
    elapsed = time.time() - start
    summary = {
        "description": "Stage 0: pre-compute and cache target Jacobians for exp16-19. Pairs = runtime re-selection (full-attention small-model layers + 0_0 + last, contract 0.1/0.2). For each layer pair (l_L, l_S) transports J_L to small-model space via J_tgt = J_L @ E_L.T @ E_S @ G_S_inv, then gauge-free variants J_tgt_p / J_S_p via softmax weighting. Each per-sample cache holds J targets, gauge-free p over token_set, p_L_full (full-vocab p_L for exp18), token_set, and input_ids (pairing verification).",
        "layer_types": layer_types,
        "pairs": [[a, b] for a, b in pairs],
        "n_samples": _guard_n,
        "K": _guard_k,
        "device_L": device_L,
        "device_S": device_S,
        "elapsed_seconds": elapsed,
    }
    summary_path = os.path.join(targets_dir, "stage0_summary.json")
    summary["params"] = {
        "large_model": large_model,
        "small_model": small_model,
        "device_L": device_L,
        "device_S": device_S,
        "max_length": max_length,
        "min_length": min_length,
    }
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print("\n=== Stage 0 complete ===")
    print(f"Pairs cached: {len(done_pairs)}/{len(pairs)}")
    print(f"Total time: {elapsed:.0f}s ({elapsed / 60:.1f} min)")
    print(f"Summary: {summary_path}")


In [ ]:
# ---------- run exp16_stage0_targets ----------
import gc
# Hard cleanup from previous kernel version (same session, different code)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
import inspect
sig = inspect.signature(main_16stage0targets)
kwargs = {}
params = list(sig.parameters)
if 'small_model' in params: kwargs['small_model'] = SMALL
if 'large_model' in params: kwargs['large_model'] = LARGE
if 'model' in params: kwargs['model'] = SMALL
if 'device_L' in params: kwargs['device_L'] = device_L
if 'device_S' in params: kwargs['device_S'] = device_S
if 'device' in params: kwargs['device'] = device_S
if 'results_dir' in params: kwargs['results_dir'] = RES
if 'targets_dir' in params: kwargs['targets_dir'] = os.path.join(RES, 'targets')
# Remove conflicting keys: if both model and small_model, prefer specific
if 'small_model' in kwargs and 'model' in kwargs: del kwargs['model']
# Memory diagnostic before experiment
if torch.cuda.is_available():
    print(f'cuda:0 alloc={torch.cuda.memory_allocated(0)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(0)/1024**3:.1f}GiB')
    print(f'cuda:1 alloc={torch.cuda.memory_allocated(1)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(1)/1024**3:.1f}GiB')
t0 = time.time()
try:
    main_16stage0targets(**kwargs)
    elapsed = time.time() - t0
    print(f"[OK] exp16_stage0_targets {elapsed:.0f}s")
    # Persist models (in case experiment loaded from HF but didn't save)
    try:
        dst_s = os.path.join(WORKING, "qwen35-08b")
        dst_l = os.path.join(WORKING, "qwen35-4b")
        if os.path.isdir(dst_s):
            print(f"Small model persisted at {dst_s}")
        if os.path.isdir(dst_l):
            print(f"Large model persisted at {dst_l}")
    except Exception:
        pass
except Exception as e:
    print(f"[FAIL] exp16_stage0_targets: {e}")
    traceback.print_exc()
print("\nResults:")
if os.path.isdir(RES):
    try:
        for fn in sorted(os.listdir(RES)):
            print(f"  {fn} {os.path.getsize(os.path.join(RES, fn)):,} bytes")
    except OSError:
        pass
